In [2]:
import numpy as np
import pandas as pd
from configs import (TABLES, ROOT, HEAVY, MIN_GENES, DROP_CONTEXTS,
                     DROP_RELIABILITY, POSITIVE, NEGATIVE, SPLIT_FRACTIONS)

In [3]:
HPA_ZIP = "../external/normal_ihc_data.tsv.zip"

SEED = 20261105

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)

key = ["Gene", "Tissue", "Cell type"]

In [4]:
hpa = pd.read_csv(HPA_ZIP, sep="\t")
print("rows read", len(hpa))

rows read 1199675


In [5]:
hpa = hpa.dropna(subset=["Level"])
print("rows with a level", len(hpa))

rows with a level 1197841


In [6]:
print(pd.crosstab(
    hpa["Cell type"].str.contains("Gradient", regex=False).rename("gradient_context"),
    hpa["Level"].isin(["Ascending", "Descending"]).rename("ascending_or_descending"),
))

ascending_or_descending    False  True 
gradient_context                       
False                    1197563      0
True                           0    278


In [9]:
hpa = hpa[hpa["Level"].isin(POSITIVE + NEGATIVE)]
print("\nrows with a usable level", len(hpa))


rows with a usable level 1197552


In [10]:
shared = hpa.groupby("Gene name")["Gene"].nunique()
print("\n== symbols shared by more than one Ensembl gene")
print(shared[shared > 1].to_string())


== symbols shared by more than one Ensembl gene
Gene name
HERC3     2
MATR3     2
NPIPA9    2
PINX1     2
PRODH     2
TBCE      2


In [11]:
shared

Gene name
A1BG      1
A1CF      1
A2M       1
A2ML1     1
A4GNT     1
         ..
ZXDA      1
ZXDB      1
ZYG11B    1
ZYX       1
ZZZ3      1
Name: Gene, Length: 13472, dtype: int64

In [17]:
contexts = hpa.groupby(["Tissue", "Cell type"])["Gene"].nunique().rename("n_genes").reset_index()
drop_set = set(DROP_CONTEXTS)
contexts["dropped_by_rule"] = [(t, c) in drop_set for t, c in zip(contexts["Tissue"], contexts["Cell type"])]
contexts["kept"] = (contexts["n_genes"] >= MIN_GENES) & ~contexts["dropped_by_rule"]
contexts = contexts.sort_values("n_genes", ascending=False)
contexts.to_csv(TABLES / "01_contexts.csv", index=False)

In [23]:
kept = contexts[contexts["kept"]]
print("\n== contexts", len(contexts), "kept", len(kept))
print("dropped by rule")
print(contexts[contexts["dropped_by_rule"]].to_string(index=False))
print("shallowest kept")
print(kept.tail(5).to_string(index=False))
print("deepest excluded by depth")
print(contexts[~contexts["kept"] & ~contexts["dropped_by_rule"]].head(3).to_string(index=False))


== contexts 248 kept 75
dropped by rule
         Tissue Cell type  n_genes  dropped_by_rule  kept
Cerebral cortex  neuropil    13460             True False
shallowest kept
     Tissue                   Cell type  n_genes  dropped_by_rule  kept
      Colon   peripheral nerve/ganglion    10765            False  True
   Placenta              decidual cells    10323            False  True
Soft tissue                chondrocytes     8255            False  True
     Testis cells in seminiferous ducts     8035            False  True
      Ovary              follicle cells     7755            False  True
deepest excluded by depth
Tissue                    Cell type  n_genes  dropped_by_rule  kept
Testis                sertoli cells     5441            False False
Testis elongated or late spermatids     5441            False False
Testis      pachytene spermatocytes     5441            False False


In [24]:
hpa = hpa.merge(kept[["Tissue", "Cell type"]], on=["Tissue", "Cell type"])
print("\nrows in kept contexts", len(hpa))


rows in kept contexts 1016417


In [28]:
print("\n== reliability")
print("genes with more than one reliability value:", int((hpa.groupby("Gene")["Reliability"].nunique() > 1).sum()))
hpa["y"] = hpa["Level"].isin(POSITIVE).astype("int8")
print("positive rate, all reliabilities", round(hpa["y"].mean(), 4), "rows", len(hpa), "genes", hpa["Gene"].nunique())
hpa = hpa[~hpa["Reliability"].isin(DROP_RELIABILITY)]
print("positive rate, Uncertain removed ", round(hpa["y"].mean(), 4), "rows", len(hpa), "genes", hpa["Gene"].nunique())


== reliability
genes with more than one reliability value: 0
positive rate, all reliabilities 0.5484 rows 1016417 genes 13478
positive rate, Uncertain removed  0.535 rows 836906 genes 11155


In [29]:
dup = hpa[hpa.duplicated(key, keep=False)].sort_values(key + ["IHC tissue name"])
assert dup.groupby(key).size().max() == 2
paired = (
    dup.groupby(key)
    .agg(
        sample_a=("IHC tissue name", "first"), sample_b=("IHC tissue name", "last"),
        level_a=("Level", "first"), level_b=("Level", "last"),
        y_a=("y", "first"), y_b=("y", "last"),
    )
    .reset_index()
)

In [33]:
labels = (
    hpa.groupby(key, as_index=False)
    .agg(gene_name=("Gene name", "first"), y=("y", "max"))
    .rename(columns={"Gene": "gene", "Tissue": "tissue", "Cell type": "cell_type"})
)
labels["context"] = labels["tissue"] + " | " + labels["cell_type"]
labels.insert(0, "pair_id", np.arange(len(labels)))

In [35]:
hpa

,Gene,Gene name,Tissue,IHC tissue name,Cell type,Level,Reliability,y
0,ENSG00000000003,TSPAN6,Adipose tissue,Adipose tissue,adipocytes,Not detected,Approved,0
1,ENSG00000000003,TSPAN6,Adrenal gland,Adrenal gland,glandular cells,Not detected,Approved,0
2,ENSG00000000003,TSPAN6,Appendix,Appendix,glandular cells,Medium,Approved,1
3,ENSG00000000003,TSPAN6,Appendix,Appendix,lymphoid tissue,Not detected,Approved,0
4,ENSG00000000003,TSPAN6,Bone marrow,Bone marrow,hematopoietic cells,Not detected,Approved,0
...,...,...,...,...,...,...,...,...
1016412,ENSG00000291316,ENSG00000291316,Tonsil,Tonsil,germinal center cells,Low,Approved,1
1016413,ENSG00000291316,ENSG00000291316,Tonsil,Tonsil,non-germinal center cells,Low,Approved,1
1016414,ENSG00000291316,ENSG00000291316,Tonsil,Tonsil,squamous epithelial cells,Medium,Approved,1
1016415,ENSG00000291316,ENSG00000291316,Urinary bladder,Urinary bladder,urothelial cells,High,Approved,1
